In [3]:
import os

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import yaml

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
noise_levels = config["noise"]["sigmas"]
batch_size = config["training"]["batch_size"]

In [5]:
import torch
import numpy as np
import random

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)



In [6]:
import torch.nn.functional as F

def cosine_similarity(a, b):
    a = F.normalize(a, dim=1)
    b = F.normalize(b, dim=1)
    return (a * b).sum(dim=1).mean().item()

In [7]:
def evaluate_embedding_stability(model, loader, device, sigmas):
    model.eval()
    results = {}

    for sigma in sigmas:
        sims = []

        for images, _ in loader:
            images = images.to(device)

            noise = torch.randn_like(images) * sigma
            noisy_images = images + noise

            with torch.no_grad():
                clean_feat, _ = model(images)
                noisy_feat, _ = model(noisy_images)

            # 🔥 Use your cosine_similarity function here
            sim = cosine_similarity(clean_feat, noisy_feat)
            sims.append(sim)

        results[sigma] = sum(sims) / len(sims)
        print(f"Sigma {sigma}: {results[sigma]:.4f}")

    return results

In [8]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"

image_size = config["data"]["image_size"]

test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        config["normalization"]["mean"],
        config["normalization"]["std"]
    )
])

base_dir = "/content/images"
test_dir = os.path.join(base_dir, "testing")

test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
# eff_model, res_model, swin_model already defined (NOTEBOOK 3 CODES)

import os
import yaml
import torch

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

import torch.nn as nn
from torchvision import models

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"
train_dir = "/content/images/training"
num_classes = len(os.listdir(train_dir))

print("Number of classes:", num_classes)

#efficientnet
def get_efficientnet(num_classes):
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")

    in_features = model.classifier[1].in_features

    # Replace classifier
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False

    return model, in_features


def get_resnet(num_classes):
    model = models.resnet50(weights="IMAGENET1K_V1")

    in_features = model.fc.in_features

    model.fc = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False

    return model, in_features


def get_swin(num_classes):
    model = models.swin_v2_t(weights="IMAGENET1K_V1")

    in_features = model.head.in_features

    model.head = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

    return model, in_features






Using device: cuda
replace /content/images/testing/Aedes Aegypti/Aedes Aegypti_401.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Number of classes: 6


In [10]:
class ModelWithEmbedding(nn.Module):
    def __init__(self, model, model_name):
        super().__init__()
        self.model = model
        self.model_name = model_name

    def forward(self, x):
        if self.model_name == "resnet":
            # Extract features before FC
            features = self.model.avgpool(self.model.layer4(
                        self.model.layer3(
                        self.model.layer2(
                        self.model.layer1(
                        self.model.maxpool(
                        self.model.relu(
                        self.model.bn1(
                        self.model.conv1(x)))))))))
            features = torch.flatten(features, 1)
            logits = self.model.fc(features)

        elif self.model_name == "efficientnet":
            features = self.model.features(x)
            features = self.model.avgpool(features)
            features = torch.flatten(features, 1)
            logits = self.model.classifier(features)

        elif self.model_name == "swin":

            x=self.model.features(x)

            x=self.model.norm(x)

            features = x.mean(dim=(1,2))

            logits = self.model.head(features)
        return features, logits

In [11]:
eff_model= get_efficientnet(num_classes)
res_model= get_resnet(num_classes)
swin_model= get_swin(num_classes)

eff_model = ModelWithEmbedding(eff_model, "efficientnet").to(DEVICE)
res_model = ModelWithEmbedding(res_model, "resnet").to(DEVICE)
swin_model = ModelWithEmbedding(swin_model, "swin").to(DEVICE)

print("Models initialized.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 171MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s]


Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth


100%|██████████| 109M/109M [00:00<00:00, 148MB/s] 


Models initialized.


In [12]:
import torch

eff_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/efficientnet.pth"))
res_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/resnet.pth"))
swin_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/swin.pth"))


sigmas = [0.0, 0.05, 0.1, 0.15, 0.2]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

eff_model.to(device).eval()
res_model.to(device).eval()
swin_model.to(device).eval()

def set_eval(m):
    if isinstance(m, torch.nn.BatchNorm2d) or isinstance(m, torch.nn.BatchNorm1d):
        m.eval()

eff_model.apply(set_eval)
res_model.apply(set_eval)
swin_model.apply(set_eval)

import torch
import os


images, _ = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    f1, _ = eff_model(images)
    f2, _ = eff_model(images)

sim = cosine_similarity(f1, f2)
print("Sanity check similarity:", sim)




Using device: cuda
Using device: cuda
Sanity check similarity: 1.0


In [13]:
print("\nEmbedding Stability - EfficientNet")
eff_embed = evaluate_embedding_stability(eff_model, test_loader, device, sigmas)

print("\nEmbedding Stability - ResNet")
res_embed = evaluate_embedding_stability(res_model, test_loader, device, sigmas)

print("\nEmbedding Stability - Swin")
swin_embed = evaluate_embedding_stability(swin_model, test_loader, device, sigmas)


Embedding Stability - EfficientNet
Sigma 0.0: 1.0000
Sigma 0.05: 0.1343
Sigma 0.1: -0.0030
Sigma 0.15: -0.0470
Sigma 0.2: -0.0849

Embedding Stability - ResNet
Sigma 0.0: 1.0000
Sigma 0.05: 0.8810
Sigma 0.1: 0.8091
Sigma 0.15: 0.7657
Sigma 0.2: 0.7324

Embedding Stability - Swin
Sigma 0.0: 1.0000
Sigma 0.05: 0.9076
Sigma 0.1: 0.8311
Sigma 0.15: 0.7667
Sigma 0.2: 0.7127
